In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
import timm 
from transformers import AutoModel
from sklearn.cluster import DBSCAN
from wildlife_datasets.datasets import AnimalCLEF2026
from wildlife_tools.features import DeepFeatrues
from wildlife_tools.similarity import ConsineSimilarity


### Visualizing Data

In [ ]:
root = '/kaggle/input/animal-clef-2026'
dataset_full = AnimalCLEF2026(
    root,
    transform=None,
    load_label=True,
    factorize_label=True,
    check_files=False
)

In [ ]:
dataset_full.metadata.head()

In [ ]:
dataset_full.metadata[['dataset', 'split']].value_counts(sort=False)

In [ ]:
dataset_full = dataset_full.get_subset(dataset_full.df['split'] == 'test')

datasets = {}
for name in dataset_full.metadata['dataset'].unique():
    datasets[name] = dataset_full.get_subset(dataset_full.df['dataset'] == name)

datasets

In [ ]:
for dataset in datasets.values():
    dataset.plot_grid(n_rows=3, n_cols=4, rotate=False);

In [ ]:
device = 'cuda'
batch_size=32

similarities = {}
for name, dataset in datasets.items():
    # Select the model for feature extraction
    if name in ['SalamanderID2025', 'SeaTurtleID2022']:
        model = timm.creat_model("hf-hub:BVRA/MegaDescriptor-L-284", pretrained=True).eval()
        size = 384
    elif name in ['LynxID2025', 'TexasHornedLizards']:
        model = AutoModel.from_pretrained("conservationxlabs/miewid-msv3", trust_remote_code=True)
        size = 512
    else:
        raise ValueError('Name does not exist')
    
    # Set the extractor and transform for the images
    matcher = ConsineSimilarity()
    extractor = DeepFeatrues(model=model, device=device, batch_size=batch_size)
    transform = T.Compose([
        T.Resize(size=(size, size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    # Set the transform for images
    dataset.set_transform(transform)
    # Extract features
    features = extractor(dataset)
    # Compute the similarity matrix
    similarity = matcher(features, features)
    similarities[name] = similarity

In [ ]:
print(f'Dataset {name} with {len(dataset)} images.')
print(f'Features have size {features.features.shape}.')
print(f'Similarity matrix has shape {similarity.shape}.')

In [ ]:
def relabel_negatives(labels):
    labels = np.array(labels)
    neg_indices = np.where(labels = -1)[0]
    new_labels = np.arange(labels.max()+1, labels.max()+1+len(neg_indices))
    labels[neg_indices] = new_labels
    return labels

def run_DBSCAN(similarity, eps):
    # Convert similarity (high is good) to distance (small is good)
    distance = (np.max(similarity) - np.maximum(similarity, 0)) / np.max(similarity)
    # Obtain predictions
    clustering = DBSCAN(eps=eps, metric='precomputed', min_samples=2)
    clusters = clustering.fit(distance)
    # Relabel -1 clusters into separate clusters
    return relabel_negatives(clusters.labels_)

In [ ]:
results = None
eps_opt = {
    'LynxID2025'
}